# Chapter 9 · From the real workflow to the specification

Five new functions for the store, asked for in four forms: as a plain request, with the notes of the conversations with the people who do the work, with a specification the agent writes from those notes, and with the store's specification. Each function has two kinds of hidden tests: the ones for what the request asks, and the criterion tests, for the decisions the program does not make.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. Sections 6 and 7 can be done without a GPU.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model, the repository and the tools

In [ ]:
!pip install -q transformers==5.19.0  # the versions the book was checked with

In [ ]:
"""Chapter 9: from the real workflow to the specification."""
import ast
import importlib
import io
import json
import re
import shutil
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
ROOT = Path.cwd()                          # where data/ and shared/ are
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")
TASKS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
         "T17", "T18", "T19", "T20"]        # the single-session tasks
MAX_STEPS = 30             # model calls per task
MAX_TOKENS_STEP = 600      # new tokens per call
INPUT_LIMIT = 12_000       # tokens of the conversation: more does not fit on the T4
INTACT_RESULTS = 2         # the latest tool results are never trimmed
MAX_REJECTIONS = 3         # times the gate sends the work back
GREEDY = {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
SAMPLING = {"do_sample": True, "temperature": 0.7, "top_p": 0.8,
            "top_k": 20}           # the values Qwen recommends for this model
decoding = dict(GREEDY)            # the one generate() uses; run_all() changes it
rt = None                  # the tools module, after prepare()



def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; otherwise, on CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def prepare():
    """The North Cycles repository, its tasks and the agent's tools. If they are
    missing (in Colab), they are downloaded from the book's repository."""
    global rt
    if not (ROOT / "shared" / "repo_tools.py").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith(("en/data/", "en/shared/")) and name[-1] != "/":
                destination = ROOT / relative[len("en/"):]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    sys.path.insert(0, str(ROOT / "shared"))
    rt = importlib.import_module("repo_tools")
    print(f"Repository in {rt.REPO}: {len(rt.files_in(rt.REPO))} files")

In [ ]:
tok, model = load()
prepare()

## 2. The model

In [ ]:
def template(tok, messages):
    """The conversation as text, with the tools, in the model's chat template."""
    return tok.apply_chat_template(messages, tools=rt.SCHEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generate(tok, model, text):
    """One answer from the model, with the current decoding: greedy or with
    sampling. Returns the text and the input and output tokens."""
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, **decoding, max_new_tokens=MAX_TOKENS_STEP,
                                prefill_chunk_size=1024, pad_token_id=tok.eos_token_id)
    n_input = inputs["input_ids"].shape[1]
    new = output[0, n_input:].cpu()
    del inputs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # each step has a different length: without this,
        #                           reserved memory grows until the GPU is full
    return tok.decode(new, skip_special_tokens=True).strip(), n_input, len(new)


def read_calls(answer):
    """The calls {name, arguments} that the model writes between <tool_call> and
    </tool_call>. A call that cannot be understood gets name None."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", answer, re.S):
        try:
            data = json.loads(block.strip(), strict=False)
        except ValueError:
            data = None
        if isinstance(data, dict) and "name" in data:
            arguments = data.get("arguments", {})
            calls.append({"name": data["name"],
                          "arguments": arguments if isinstance(arguments, dict)
                          else {}})
        else:
            calls.append({"name": None, "arguments": {}})
    return calls


def trim_conversation(tok, messages):
    """If the conversation goes over INPUT_LIMIT tokens, replaces the oldest tool
    results with a short notice. Returns the text for the model and how many it
    has trimmed (None if even that does not make it fit)."""
    text = template(tok, messages)
    trimmed = 0
    candidates = [m for m in messages if m["role"] == "tool"
                  and not m.get("trimmed")][:-INTACT_RESULTS or None]
    while len(tok(text)["input_ids"]) > INPUT_LIMIT:
        if not candidates:
            return text, None
        message = candidates.pop(0)
        message["content"] = json.dumps(
            {"notice": "Old result trimmed to save memory; ask for it again if you "
                       "need it.",
             "start": message["content"][:150]}, ensure_ascii=False)
        message["trimmed"] = True
        trimmed += 1
        text = template(tok, messages)
    return text, trimmed

## 3. The rules and the contract, which now also overrides R5 and R6

In [ ]:
SYSTEM = (
    "You are a coding agent working on the repository of North Cycles, a bike "
    "store in Portland, Oregon. It is a management program in Python with several "
    "modules (orders, returns, stock, invoices, notices…), its tests in the tests/ "
    "folder and some sample data. You will get a request from someone at the store, "
    "and you have to solve it by changing the code with the tools: list files, read "
    "line ranges, search, edit by exact replacement, run the tests and a few allowed "
    "commands. Work step by step, one tool at a time. When you are done, answer "
    'without calling any tool, starting with "DONE:" and explaining in one or two '
    "sentences what you changed. If you cannot finish, start with \"NOT DONE:\" and "
    "explain why.")


AGENTS_RULES = """# AGENTS.md · North Cycles

Repository rules. Each one is checked at the end of your work.

R1. Before editing a file, read it with read_file.
R2. Do not change anything in data/.
R3. Change docs/ only if the request asks to change the terms and conditions.
R4. Do not delete or change existing tests. Exception: if the request changes a
    business rule, update the tests of the old rule and say so in your answer.
R5. Add your tests to the module's test file (tests/test_shipping.py for shipping.py):
    replace the line `if __name__ == "__main__":` with a new class containing your
    test, followed by that same line. Import what you need inside the test.
R6. Do not create new modules outside tests/.
    Exception to R5 and R6: create the files the contract names in its allowed
    paths, such as a new module and its test file.
R7. "DONE" requires: changes to the program code, a new check in tests/ and
    run_tests green after your last change.
R8. Your final answer names every file you changed.
"""


ASKS_DOCS = re.compile(r"(updat\w*|and in the)\s[^.]{0,40}terms\s+and\s+conditions",
                       re.I)


MENTIONS_TESTS = r"run_tests|(?<![\w/])tests?\b(?![/_])"  # "test", not a file name


def diff_lines(diff):
    """The added and removed lines of each file of a unified diff:
    {path: {"+": [...], "-": [...]}}."""
    files, current = {}, None
    for line in diff.splitlines():
        if line.startswith("+++ b/"):
            current = files.setdefault(line[6:], {"+": [], "-": []})
        elif line.startswith("--- a/") or current is None:
            continue
        elif line[:1] in "+-" and line[1:].strip():
            current[line[0]].append(line[1:])
    return files


def check_rules(task, log, changes, answer, require_test=True,
                allows_docs=None, allows_create=()):
    """The broken rules, as {rule: evidence}. It looks at what has really changed
    (the diff), the log of calls and the final answer. With "require_test" False,
    R7 does not ask for a new check; "allows_docs", if given, decides R3 instead of
    the request; the files in "allows_create" are exceptions to R5 and R6."""
    if allows_docs is None:
        allows_docs = bool(ASKS_DOCS.search(task["request"]))
    broken = {}
    lines = diff_lines(changes["diff"])
    read = set()
    for r in log:
        path = r["arguments"].get("path", "")
        if r["tool"] == "read_file" and r["ok"]:
            read.add(path)
        elif r["tool"] == "edit" and r["ok"] and path not in read:
            broken.setdefault("R1", f"you edited {path} without reading it first")
    for path in changes["changed"]:
        if path.startswith("data/"):
            broken["R2"] = f"you changed {path}"
        if path.startswith("docs/") and not allows_docs:
            broken["R3"] = f"you changed {path} and the request does not ask for it"
        if path in allows_create:
            continue
        if path.startswith("tests/") and path in changes["new"]:
            broken["R5"] = f"you created {path}; tests go in the module's test file"
        if not path.startswith(("tests/", "docs/", "data/")) and path in changes["new"]:
            broken["R6"] = f"you created the module {path}"
    removed = [r for r, l in lines.items() if r.startswith("tests/") and l["-"]]
    if removed and not re.search(MENTIONS_TESTS, answer or "", re.I):
        broken["R4"] = (f"you changed existing tests in {', '.join(removed)} "
                        "and your answer does not say so")
    if answer and says_done(answer):
        program = [r for r in changes["changed"]
                   if not r.startswith(("tests/", "docs/", "data/"))]
        new_test = any(re.search(r"def test_|assert", l)
                       for r, l2 in lines.items() if r.startswith("tests/")
                       for l in l2["+"])
        edits = [i for i, r in enumerate(log)
                 if r["tool"] in ("edit", "create_file") and r["ok"]]
        after = log[edits[-1] + 1:] if edits else log
        runs = [r for r in after if r["tool"] == "run_tests"]
        missing = [text for met, text in (
            (program, "there are no changes to the program code"),
            (new_test or not require_test,
             "there is no new check in tests/"),
            (runs and runs[-1]["result"].get("all_pass"),
             "there is no green test run after your last change"))
            if not met]
        if missing:
            broken["R7"] = "; ".join(missing)
        unnamed = [r for r in changes["changed"]
                   if Path(r).name not in answer]
        if unnamed:
            broken["R8"] = f"your answer does not name {', '.join(unnamed)}"
    return broken


def says_done(answer):
    """True if the final answer says the task is done. An empty answer says
    nothing."""
    if not answer.strip():
        return False
    if re.search(r"NOT\s+DONE", answer, re.I):
        return False
    if re.search(r"DONE", answer):
        return True
    return not re.search(r"\bI (couldn[’']t|could not|can[’']t|cannot|was not able)"
                         r"|\bunable\b|not possible|\bI need\b|\?", answer, re.I)


class ToolsWithRules:
    """The agent's tools with the process rules checked before every edit: if the
    edit breaks R1, R2, R3, R5 or R6, it is not made and the agent gets the rule
    back. The other calls go through as they are."""

    def __init__(self, tools, task):
        self.tools = tools
        self.log = tools.log
        self.task = task
        self.read = set()

    def broken_rule(self, name, path):
        """The rule that editing or creating "path" would break, or None."""
        exists = self.tools.workspace.path(path).exists()
        if path.startswith("data/"):
            return "R2: do not change anything in data/."
        if path.startswith("docs/") and not self.allows_docs():
            return "R3: the request does not ask to change the terms and conditions."
        if name == "create_file" and self.allows_create(path):
            return None
        if name == "create_file" and path.startswith("tests/"):
            return ("R5: do not create test files; add your test to the module's test "
                    "file, replacing the line if __name__ == \"__main__\":")
        if name == "create_file" and not exists:
            return "R6: do not create new modules outside tests/."
        if name == "edit" and exists and path not in self.read:
            return f"R1: read {path} with read_file before editing it."
        return None

    def allows_docs(self):
        """True if the request asks to change the terms and conditions (R3)."""
        return bool(ASKS_DOCS.search(self.task["request"]))

    def allows_create(self, path):
        """Without a contract, no new file is an exception to R5 or R6."""
        return False

    def call(self, name, arguments):
        """Like Tools.call, but checking the rules before editing."""
        path = str((arguments or {}).get("path", ""))
        if name in ("edit", "create_file"):
            rule = self.broken_rule(name, path)
            if rule:
                result = {"error": "Edit rejected by AGENTS.md. " + rule}
                self.log.append({"tool": name, "arguments": arguments,
                                 "ok": False, "seconds": 0,
                                 "result": result})
                return result
        result = self.tools.call(name, arguments)
        if name == "read_file" and "error" not in result:
            self.read.add(path)
        return result


def contract_text(contract):
    """A contract (like data/tasks/Txx/contract.json) written for the agent: goal,
    paths it may change, forbidden paths and numbered acceptance criteria (C1,
    C2…)."""
    criteria = "\n".join(f"C{i}. {c}" for i, c in
                         enumerate(contract["acceptance_criteria"], 1))
    return ("Task contract\n"
            f"Goal: {contract['goal']}\n"
            f"You may change: {', '.join(contract['allowed_paths'])}\n"
            f"Don't touch: {', '.join(contract['forbidden_paths'])}\n"
            f"Acceptance criteria:\n{criteria}\n"
            "The contract overrides AGENTS.md: if it forbids tests/, don't add tests.")


def forbids_tests(contract):
    """True if the contract forbids changing tests/ (for example, a refactoring
    that must pass the tests without touching them)."""
    return rt.path_matches("tests/x.py", contract["forbidden_paths"])


def check_contract(contract, changes, answer):
    """What breaks the contract at the end: files outside the allowed paths or in
    the forbidden ones, and criteria the answer does not explain how it meets."""
    broken = {}
    paths = rt.check_scope(changes["changed"], contract)
    outside = sorted(set(paths["out_of_scope"]) | set(paths["forbidden"]))
    if outside:
        broken["Scope"] = (f"you changed {', '.join(outside)}, which the contract "
                           "does not allow")
    numbers = range(1, len(contract["acceptance_criteria"]) + 1)
    unexplained = [f"C{i}" for i in numbers
                   if not re.search(rf"\bC{i}\b", answer or "")]
    if unexplained:
        broken["Criteria"] = ("your answer does not say how it meets "
                              + ", ".join(unexplained)
                              + " (cite each criterion by its number)")
    return broken


class ToolsWithContract(ToolsWithRules):
    """The tools with the rules of chapter 2 and, on top of them, the contract:
    nothing can be edited or created outside the allowed paths."""

    def __init__(self, tools, task, contract):
        super().__init__(tools, task)
        self.contract = contract

    def allows_docs(self):
        """The contract overrides R3: docs/ can be changed if the contract allows it."""
        return rt.path_matches("docs/terms_and_conditions.md",
                               self.contract["allowed_paths"])

    def allows_create(self, path):
        """The contract overrides R5 and R6: a file can be created if the contract
        names it exactly among its allowed paths."""
        return path in self.contract["allowed_paths"]

    def broken_rule(self, name, path):
        """The rule or the part of the contract the edit would break, or None."""
        contract = self.contract
        if rt.path_matches(path, contract["forbidden_paths"]):
            return f"Contract: {path} is among the paths you must not touch."
        if not rt.path_matches(path, contract["allowed_paths"]):
            return (f"Contract: {path} is not among the paths you may change ("
                    + ", ".join(contract["allowed_paths"]) + ").")
        return super().broken_rule(name, path)

## 4. Feedback, gate, reviewer and frame from chapters 5 to 7

In [ ]:
MAX_FAILURES = 3   # failures summarized for the agent


def summarize_failures(output, folder):
    """The failures of a unittest run, one per test: which test fails, the lines of
    the repository the error goes through (without Python's own) and the last line,
    the one with the error. Paths are left relative to the copy of the repository."""
    failures = []
    prefix = str(folder) + "\\" if "\\" in str(folder) else str(folder) + "/"
    for block in re.split(r"^={50,}$", output, flags=re.M)[1:]:
        header = re.search(r"^(FAIL|ERROR): (\S+)", block, re.M)
        if not header:
            continue
        lines = block.split("-" * 70)[1] if "-" * 70 in block else block
        at = []
        for m in re.finditer(r'File "([^"]+)", line (\d+), in (\S+)\n\s+(.+)', lines):
            if m.group(1).startswith(prefix):
                path = m.group(1)[len(prefix):].replace("\\", "/")
                at.append(f"{path}:{m.group(2)} ({m.group(3)}): {m.group(4).strip()}")
        error = [l for l in lines.strip().splitlines() if l.strip()][-1:]
        failures.append({"test": f"{header.group(1)} {header.group(2)}",
                         "where": at[-3:], "error": error[0].strip() if error else ""})
    return failures[:MAX_FAILURES]


class ToolsWithFeedback(ToolsWithContract):
    """The tools with the contract and, besides, feedback: after each successful
    edit the tests are run and the result of the edit carries a summary of the
    failures. run_tests returns the same summary."""

    def tests(self):
        """The repository's tests, summarized for the agent."""
        result = self.tools.run_tests()
        summary = {"all_pass": result["all_pass"],
                   "result": result["result"]}
        if not result["all_pass"]:
            summary["failures"] = summarize_failures(result["output"],
                                                     self.tools.workspace.folder)
        return summary

    def call(self, name, arguments):
        """Like ToolsWithContract.call, with the tests after each edit."""
        if name == "run_tests":
            result = self.tests()
            self.log.append({"tool": name, "arguments": {}, "ok": True,
                             "seconds": 0, "result": result})
            return result
        result = super().call(name, arguments)
        if name in ("edit", "create_file") and "error" not in result:
            result = dict(result, tests_after_edit=self.tests())
        return result


def code_without_docstrings(text):
    """The syntax tree of a program, as text, without its docstrings. Comments never
    reach the tree, so two versions that differ only in comments or docstrings give
    the same result. None if it cannot be parsed."""
    try:
        tree = ast.parse(text)
    except SyntaxError:
        return None
    for node in ast.walk(tree):
        body = getattr(node, "body", None)
        if (isinstance(body, list) and body and isinstance(body[0], ast.Expr)
                and isinstance(body[0].value, ast.Constant)
                and isinstance(body[0].value.value, str)):
            node.body = body[1:] or [ast.Pass()]
    return ast.dump(tree)


def changes_code(workspace, path):
    """True if the file has changed in something other than comments and docstrings."""
    original, new = workspace.source / path, workspace.path(path)
    if not path.endswith(".py") or not original.exists() or not new.exists():
        return True
    before = code_without_docstrings(original.read_text(encoding="utf-8"))
    after = code_without_docstrings(new.read_text(encoding="utf-8"))
    return before is None or after is None or before != after


def verification_gate(workspace, contract):
    """What is missing to accept the task, looking at the repository and not at what
    the agent says: code changes that are not only comments, a new test that fails
    with the original code (if the contract allows touching tests/) and all the
    tests green. Returns a list of sentences; empty if the work can be accepted."""
    changes = rt.differences(workspace.source, workspace.folder)
    lines = diff_lines(changes["diff"])
    missing = []
    code = [r for r in changes["changed"]
            if not r.startswith(("tests/", "docs/", "data/"))]
    if not any(changes_code(workspace, r) for r in code):
        missing.append("the program code has not changed (comments don't count)")
    new_tests = [r for r, l in lines.items() if r.startswith("tests/")
                 and any(re.search(r"def test_|assert", x) for x in l["+"])]
    if not forbids_tests(contract):
        if not new_tests:
            missing.append("there is no new test in tests/")
        else:
            with rt.Workspace() as original:    # the new tests, with the old code
                for path in new_tests:
                    destination = original.path(path)
                    destination.write_text(workspace.path(path).read_text("utf-8"),
                                           encoding="utf-8")
                before = rt.Tools(original).run_tests()
            if before["all_pass"]:
                missing.append("your new tests also pass with the original code, so "
                               "they don't check the change the request asks for")
    now = rt.Tools(workspace).run_tests()
    if not now["all_pass"]:
        failures = summarize_failures(now["output"], workspace.folder)
        missing.append("the tests don't pass: "
                       + "; ".join(f"{f['test']} → {f['error']}" for f in failures))
    return missing

In [ ]:
MAX_REVIEWER_STEPS = 10  # model calls in each review


MAX_REVIEWS = 2          # times the reviewer can send the work back


ESCALATED = ("NOT DONE: the reviewer has rejected the change twice; it is waiting "
             "for a person to review it.")


READ_ONLY = ("list_files", "read_file", "search", "run_tests")


REVIEWER_SYSTEM = (
    "You are the change reviewer of the North Cycles repository, a bike store in "
    "Portland, Oregon. Another agent has made a change to solve a request and says "
    "it is done. Your job is to check whether the change really solves the "
    "request. You can list, read and search files and run the tests, but you "
    "cannot change anything. Use one tool at a time. When you are done, answer "
    "without calling any tool, with only a JSON.")


ASK_FREE = """Review this change.

Request:
{request}

Change (diff):
{diff}

What the author says:
{answer}

At the end, answer with this JSON:
{{"approved": true or false, "comments": ["…"]}}"""


ASK_RUBRIC = """Review this change criterion by criterion.

Request:
{request}

Acceptance criteria:
{criteria}

Change (diff):
{diff}

What the author says:
{answer}

For each criterion, look in the code (not in what the author says) to see whether
it is met, and give the evidence: file and line, or the test that checks it. Also
check whether the change touches anything the request does not ask for. At the end,
answer with this JSON:
{{"criteria": [{{"id": "C1", "met": true or false, "evidence": "…"}}],
  "beyond_request": ["…"], "approved": true or false}}
Approve only if every criterion is met."""


class ReadOnlyTools:
    """The reviewer's tools: the usual ones, but only those that do not change
    anything. Any other one is rejected."""

    def __init__(self, tools):
        self.tools = tools
        self.log = []

    def call(self, name, arguments):
        """Like Tools.call, only for the reading tools."""
        if name not in READ_ONLY:
            result = {"error": f'The reviewer cannot use "{name}": only '
                               + ", ".join(READ_ONLY) + "."}
        else:
            result = self.tools.call(name, arguments)
        self.log.append({"tool": name, "ok": "error" not in result})
        return result


def read_verdict(answer):
    """The last JSON object of the reviewer's answer, or None if there is none
    with the key "approved"."""
    for piece in reversed(re.findall(r"\{.*\}", answer, re.S)):
        try:
            verdict = json.loads(piece, strict=False)
        except ValueError:
            continue
        if isinstance(verdict, dict) and "approved" in verdict:
            return verdict
    return None


def review(tok, model, task, contract, workspace, author_answer, rubric, verbose=False):
    """One review: a separate agent, with its own conversation and read-only tools
    on the copy of the repository, that returns its verdict. Without a readable
    verdict, the change is approved (the reviewer does not block for failing)."""
    changes = rt.differences(workspace.source, workspace.folder)
    criteria = "\n".join(f"C{i}. {c}" for i, c in
                         enumerate(contract["acceptance_criteria"], 1))
    request_template = ASK_RUBRIC if rubric else ASK_FREE
    messages = [{"role": "system", "content": REVIEWER_SYSTEM},
                {"role": "user", "content": request_template.format(
                    request=task["request"], criteria=criteria,
                    diff=changes["diff"][:3000], answer=author_answer[:800])}]
    tools = ReadOnlyTools(rt.Tools(workspace))
    verdict, steps = None, 0
    for steps in range(1, MAX_REVIEWER_STEPS + 1):
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            break
        answer, _, _ = generate(tok, model, text)
        calls = read_calls(answer)
        if not calls:
            verdict = read_verdict(answer)
            break
        messages.append({"role": "assistant", "content": answer.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            result = (tools.call(call["name"], call["arguments"])
                      if call["name"] else {"error": "Invalid call."})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    if verbose:
        print(f"    REVIEWER ({steps} steps): "
              f"{json.dumps(verdict, ensure_ascii=False)[:300]}")
    return {"approved": True if verdict is None else bool(verdict.get("approved")),
            "verdict": verdict, "steps": steps,
            "calls": len(tools.log)}


def comments_for_author(verdict):
    """What the reviewer objects to, in sentences for the author."""
    if not verdict:
        return "the reviewer does not approve the change"
    sentences = [f"{c.get('id', '?')} is not met: {c.get('evidence', '')}"
                 for c in verdict.get("criteria", []) if not c.get("met", True)]
    sentences += [f"beyond the request: {x}" for x in verdict.get("beyond_request", [])]
    sentences += [str(x) for x in verdict.get("comments", [])]
    return "; ".join(sentences) or "the reviewer does not approve the change"

In [ ]:
MAX_FRAME_STEPS = 10   # model calls to explore and write the frame


ASK_FRAME = """Before changing anything, understand the task. Explore the repository
with the reading tools and write the task frame in JSON, with nothing after it:
{{"goal": "one sentence",
  "facts": [{{"fact": "what you have checked", "evidence": "file:line"}}],
  "paths": ["files that will have to change"],
  "unknowns": ["what you don't know and how you would find out"]}}
Each fact must cite a file and a line that prove it.

Request:
{request}

{contract}"""


ASK_PLAN = """Now write the plan to solve it, in JSON, with nothing after it:
{{"steps": [{{"step": "what to do", "file": "where", "check": "which test shows it"}}]}}
Each step must say how you will check that it is done right."""


def read_json(answer, key):
    """The last JSON object of an answer that has the given key, or None."""
    for piece in reversed(re.findall(r"\{.*\}", answer, re.S)):
        try:
            data = json.loads(piece, strict=False)
        except ValueError:
            continue
        if isinstance(data, dict) and key in data:
            return data
    return None


def check_evidence(frame, workspace):
    """The facts of the frame whose evidence does not exist: a file that is not
    there or a line it does not have."""
    bad = []
    for fact in frame.get("facts", []):
        m = re.match(r"\s*([\w./-]+\.\w+)(?::(\d+))?", str(fact.get("evidence", "")))
        file = workspace.path(m.group(1)) if m else None
        if not m or not file.is_file():
            bad.append(f'"{fact.get("fact", "")}": {fact.get("evidence")} '
                       "does not exist")
        elif m.group(2) and int(m.group(2)) > len(
                file.read_text(encoding="utf-8").splitlines()):
            bad.append(f'"{fact.get("fact", "")}": {m.group(1)} has no line '
                       f"{m.group(2)}")
    return bad


def converse(tok, model, messages, tools, max_steps, key, verbose=False):
    """A conversation with reading tools that ends with a JSON with "key".
    Returns the JSON (or None) and the steps."""
    for steps in range(1, max_steps + 1):
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            return None, steps
        answer, _, _ = generate(tok, model, text)
        calls = read_calls(answer)
        if not calls:
            messages.append({"role": "assistant", "content": answer})
            return read_json(answer, key), steps
        messages.append({"role": "assistant", "content": answer.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            result = (tools.call(call["name"], call["arguments"])
                      if call["name"] else {"error": "Invalid call."})
            if verbose:
                print(f"  [frame {steps}] {call['name']}"
                      f"({json.dumps(call['arguments'], ensure_ascii=False)[:80]})")
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    return None, max_steps


def frame_task(tok, model, task, workspace, with_plan, verbose=False):
    """The prior phase: the agent explores without changing anything and writes the
    task frame; the program checks the evidence and, if any does not exist, sends it
    back once. With "with_plan", it also writes a plan with a check per step.
    Returns the text that is added to the request and what has happened."""
    contract = task["contract"]
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + AGENTS_RULES},
                {"role": "user", "content": ASK_FRAME.format(
                    request=task["request"], contract=contract_text(contract))}]
    reading = ReadOnlyTools(rt.Tools(workspace))
    frame, steps = converse(tok, model, messages, reading, MAX_FRAME_STEPS,
                            "goal", verbose)
    bad = check_evidence(frame, workspace) if frame else []
    if bad:
        messages.append({"role": "user", "content":
                         "This evidence does not exist: " + "; ".join(bad)
                         + ". Fix the frame and write it again in JSON."})
        fixed, more = converse(tok, model, messages, reading, 4, "goal", verbose)
        steps += more
        frame = fixed or frame
        bad = check_evidence(frame, workspace) if frame else bad
    plan = None
    if with_plan and frame:
        messages.append({"role": "user", "content": ASK_PLAN})
        plan, more = converse(tok, model, messages, reading, 2, "steps", verbose)
        steps += more
    parts = []
    if frame:
        parts.append("Task frame (written before starting):\n"
                     + json.dumps(frame, ensure_ascii=False, indent=1)[:2500])
    if plan:
        parts.append("Plan:\n" + json.dumps(plan, ensure_ascii=False, indent=1)[:1500])
    return "\n\n".join(parts), {"frame": frame, "plan": plan, "steps": steps,
                                "bad_evidence": bad}

## 5. The agent

In [ ]:
def work(tok, model, task, workspace, frame=False, plan=False, reviewer=None,
         max_steps=MAX_STEPS, verbose=False):
    """The agent of chapter 7 (contract, feedback, verification gate, reviewer and,
    if asked for, frame and plan) on the copy of the repository it is given. It is a
    generator: it hands back control after each call to the model, so that several
    agents can work in turns. When it finishes, it returns what has happened."""
    contract = task["contract"]
    request = task["request"] + "\n\n" + contract_text(contract)
    phase = {"frame": None, "plan": None, "steps": 0, "bad_evidence": []}
    if frame:
        prior, phase = frame_task(tok, model, task, workspace, with_plan=plan,
                                  verbose=verbose)
        if prior:
            request += "\n\n" + prior
        yield
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + AGENTS_RULES},
                {"role": "user", "content": request}]
    final, steps, n_in, n_out, rejections = None, 0, 0, 0, 0
    reviews = []
    base = rt.Tools(workspace)
    tools = ToolsWithFeedback(base, task, contract)
    for steps in range(1, max_steps + 1):
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            break
        answer, n_input, n_output = generate(tok, model, text)
        n_in, n_out = n_in + n_input, n_out + n_output
        yield   # here the other agents can work
        calls = read_calls(answer)
        if not calls:
            messages.append({"role": "assistant", "content": answer})
            if verbose:
                print(f"[{task['id']} {steps}] {answer}")
            if says_done(answer) and rejections < MAX_REJECTIONS:
                changes = rt.differences(workspace.source, workspace.folder)
                broken = check_rules(task, base.log, changes, answer,
                                     require_test=not forbids_tests(contract),
                                     allows_docs=tools.allows_docs(),
                                     allows_create=contract["allowed_paths"])
                broken.update(check_contract(contract, changes, answer))
                broken.pop("R7", None)
                missing = verification_gate(workspace, contract)
                if missing:
                    broken["Verification"] = "; ".join(missing)
                if broken:
                    rejections += 1
                    messages.append({"role": "user", "content":
                                     "It isn't done yet: "
                                     + "; ".join(f"{r}: {t}" for r, t in
                                                 sorted(broken.items()))
                                     + ". Keep working."})
                    continue
            if reviewer and says_done(answer):
                if len(reviews) == MAX_REVIEWS:   # it has already rejected it twice
                    final = ESCALATED
                    break
                this_review = review(tok, model, task, contract, workspace, answer,
                                     rubric=reviewer == "rubric", verbose=verbose)
                reviews.append(this_review)
                if not this_review["approved"]:
                    messages.append({"role": "user", "content":
                                     "The reviewer does not approve the change: "
                                     + comments_for_author(this_review["verdict"])
                                     + ". Fix it and answer DONE again."})
                    continue
            final = answer
            break
        thought = answer.split("<tool_call>")[0].strip()
        messages.append({"role": "assistant", "content": thought,
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            if call["name"] is None:
                result = {"error": "The tool call cannot be understood."}
            else:
                result = tools.call(call["name"], call["arguments"])
            if verbose:
                print(f"[{task['id']} {steps}] {call['name']}"
                      f"({json.dumps(call['arguments'], ensure_ascii=False)[:90]})"
                      f" -> {json.dumps(result, ensure_ascii=False)[:110]}")
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    return {"final": final, "steps": steps, "input": n_in, "output": n_out,
            "rejections": rejections, "reviews": reviews, "phase": phase,
            "log": base.log}


def run_to_end(generator):
    """Advances a generator until it finishes and returns its final value."""
    while True:
        try:
            next(generator)
        except StopIteration as end:
            return end.value

## 6. The criterion tests

Without the model: the reference solution for removing a customer passes them, and a removal that deletes the customer does not.

In [ ]:
def run_criterion(folder, task):
    """The criterion tests of a task inside a copy: the decisions the store does not
    leave to the program. They are copied into a separate folder, run and deleted,
    like the hidden ones."""
    destination = Path(folder) / "_criterion"
    shutil.copytree(task["folder"] / "criterion_tests", destination, dirs_exist_ok=True)
    try:
        code, _ = rt.run_python(["-m", "unittest", "discover", "-s", "_criterion"],
                                folder, rt.TEST_SECONDS)
    finally:
        shutil.rmtree(destination, ignore_errors=True)
    return code == 0

In [ ]:
task = rt.read_task("T31")
with rt.Workspace() as workspace:
    rt.apply_solution(rt.Tools(workspace), task["folder"] / "solution")
    print("Reference:", run_criterion(workspace.folder, task))
with rt.Workspace() as workspace:
    workspace.path("removals.py").write_text(
        "def remove_customer(customers, orders, customer_id):\n"
        "    del customers[customer_id]\n"
        "    return \"done\"\n", encoding="utf-8")
    print("Deleting the customer:", run_criterion(workspace.folder, task))

## 7. The specification

The notes and the store's specification are in each task's folder.

In [ ]:
MAX_SPECIFICATION_STEPS = 10   # model calls to write the specification


ASK_SPECIFICATION = """Before writing any code, write the specification of this
function from the request and from the notes of the conversations with the store. You
can look at the repository with the reading tools. Write it in JSON, with nothing else
at the end:
{{"goal": "one sentence",
  "does": ["what the function does, with the exact values the notes give"],
  "decisions_for_a_person": ["what the program must not decide on its own, and what it
                             does then"],
  "out_of_scope": ["what was discussed but is not part of this task"],
  "criteria": ["acceptance criteria that can be checked"]}}

Request:
{request}

Notes:
{notes}"""


def read(task, name):
    """A file from a task's folder: the notes or the specification."""
    return (task["folder"] / name).read_text(encoding="utf-8")


def specify(tok, model, task, workspace, verbose=False):
    """The chapter's prior phase: the agent reads the request and the notes, looks at
    the repository without changing anything and writes the specification. If the
    decisions for a person or the criteria are missing, it is asked for them once.
    Returns the specification (or None) and the steps."""
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + AGENTS_RULES},
                {"role": "user", "content": ASK_SPECIFICATION.format(
                    request=task["request"], notes=read(task, "notes.md"))}]
    reading = ReadOnlyTools(rt.Tools(workspace))
    specification, steps = converse(tok, model, messages, reading,
                                    MAX_SPECIFICATION_STEPS, "criteria", verbose)
    if not (specification and specification.get("decisions_for_a_person")
            and specification.get("criteria")):
        messages.append({"role": "user", "content":
                         "The specification is missing the decisions for a person "
                         "or the criteria. Read the notes again and write it in full, "
                         "in JSON."})
        other, more = converse(tok, model, messages, reading, 3, "criteria", verbose)
        steps += more
        specification = other or specification
    return specification, steps


def as_text(specification):
    """The agent's specification, written to be added to the request."""
    parts = [f"Goal: {specification.get('goal', '')}"]
    for key, title in (("does", "What it does"),
                       ("decisions_for_a_person",
                        "Decisions the program does not make"),
                       ("out_of_scope", "Out of scope")):
        parts.append(title + ":\n" + "\n".join(f"- {x}" for x in
                                               specification.get(key) or []))
    return "\n\n".join(parts)

In [ ]:
print(read(rt.read_task("T31"), "notes.md"))

## 8. Four ways of asking

By default two forms are tried with removing a customer. With `COMPLETE = True`, the four forms with the five functions. Then, the saved results of the three full runs.

In [ ]:
SPEC_TASKS = ["T31", "T32", "T33", "T34", "T35"]


AGENT = {"frame": True, "plan": True, "reviewer": "rubric"}   # the one of chapter 7


FORMS = ["plain request", "with the notes", "agent's specification",
         "with specification"]


NO_CRITERIA = ["Does what the request asks."]


def ask(task, form, specification=None):
    """The task as the agent gets it in each form: what the request carries and which
    criteria its contract has. The contract's paths are always the same."""
    request = task["request"]
    criteria = NO_CRITERIA
    if form == "with the notes":
        request += ("\n\nNotes from the conversations with the store:\n"
                    + read(task, "notes.md"))
    elif form == "agent's specification" and specification:
        request += ("\n\nSpecification (written before starting):\n"
                    + as_text(specification))
        criteria = [str(c) for c in specification["criteria"]] or NO_CRITERIA
    elif form == "with specification":
        request += "\n\n" + read(task, "specification.md")
        criteria = task["contract"]["acceptance_criteria"]
    return dict(task, request=request,
                contract=dict(task["contract"], acceptance_criteria=criteria))


def agent(tok, model, task, form, verbose=False):
    """The agent of chapter 7 in its own copy, with the task asked for in one of the
    four forms. Returns the row of results: whether it meets what the request asks
    (hidden tests) and whether it respects the store's decisions (criterion tests)."""
    start = time.time()
    with rt.Workspace() as workspace:
        specification, prior = None, 0
        if form == "agent's specification":
            specification, prior = specify(tok, model, task, workspace, verbose)
        asked = ask(task, form, specification)
        outcome = run_to_end(work(tok, model, asked, workspace, verbose=verbose,
                                  **AGENT))
        evaluation = rt.evaluate_attempt(workspace, task)
        criterion = run_criterion(workspace.folder, task)
    done = outcome["final"] is not None and says_done(outcome["final"])
    return {"task": task["id"], "form": form, "solved": evaluation["solved"],
            "criterion": criterion, "says_done": done,
            "false_done": done and not (evaluation["solved"] and criterion),
            "visible": evaluation["visible"], "hidden": evaluation["hidden"],
            "out_of_scope": evaluation["out_of_scope"],
            "rejections": outcome["rejections"], "reviews": len(outcome["reviews"]),
            "specification": specification,
            "steps": outcome["steps"] + outcome["phase"]["steps"] + prior,
            "input_tokens": outcome["input"],
            "seconds": round(time.time() - start, 1),
            "answer": outcome["final"] or "(no final answer)",
            "diff": evaluation["diff"]}


def summary(rows):
    """The totals of one form of asking."""
    return {"tasks": len(rows),
            "meets_request": sum(r["hidden"] for r in rows),
            "respects_decisions": sum(r["criterion"] for r in rows),
            "both": sum(r["hidden"] and r["criterion"] for r in rows),
            "tests_green": sum(r["visible"] for r in rows),
            "false_done": sum(r["false_done"] for r in rows),
            "steps": sum(r["steps"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def run_all(tok, model, form, tasks=SPEC_TASKS, seed=None, verbose=False):
    """One form of asking with every task: its rows and its summary. Without a seed,
    with greedy decoding; with a seed, with sampling, and each task with its own
    seed."""
    decoding.clear()
    decoding.update(GREEDY if seed is None else SAMPLING)
    rows = []
    for number, task_id in enumerate(tasks):
        if seed is not None:
            torch.manual_seed(1000 * seed + 10 * FORMS.index(form) + number)
        row = agent(tok, model, rt.read_task(task_id), form, verbose)
        rows.append(row)
        print(f"  {task_id} {'meets' if row['hidden'] else 'no   '}"
              f"  {'respects' if row['criterion'] else 'decides '}"
              f"  {'says done' if row['says_done'] else 'not said '}"
              f"  steps={row['steps']:2d}  {row['seconds']:6.1f} s", flush=True)
    return {"rows": rows, "summary": summary(rows)}

In [ ]:
# With COMPLETE = False, removing a customer is tried with two forms (about 20 minutes on Colab's T4);
# with True, the four forms with the five functions (about eight hours on the T4).
COMPLETE = False
for form in (FORMS if COMPLETE else ("plain request", "with specification")):
    print(f"== {form}")
    r = run_all(tok, model, form, tasks=SPEC_TASKS if COMPLETE else ["T31"], seed=1)
    print(r["summary"])

In [ ]:
saved = json.loads((ROOT / "data" / "results" / "chapter_9" / "results_repetitions.json").read_text(encoding="utf-8"))
for key, r in saved.items():
    print(key, r["summary"])

## Exercises

The results files the exercises cite are in `data/results/chapter_9/`, which `prepare` downloads with the data.

1. Read the notes for the backorders of sold-out bikes (`data/tasks/T34/notes.md`) and write the specification yourself before looking at the store's. Compare them: which decision had you missed? *(No GPU.)*

2. With `results_repetitions.json`, look among the functions asked for as a plain request for a decision the agent made on its own. What would have happened at the store if that code had been put into production? *(No GPU.)*

3. Compare, in the results, the specifications the agent wrote for the same function in the three runs. Do they capture the same decisions? Which one would you review, and what would you change in it? *(No GPU.)*

4. Write a criterion test for a decision in the notes that does not have one, for example that the loyalty discount is rounded to cents. Check with the reference solution that it passes. *(No GPU.)*

5. Make the reviewed specification: take the agent's specification for the expensive estimates (T32), fix it the way Evan would and give it to the agent as if it were the store's. *(GPU, about 30 minutes.)*

6. The notes carry noise: comments that have nothing to do with the task. Remove the noise from the notes of one function and ask for it with the clean notes. Does the agent use them now? *(GPU, about 30 minutes.)*

7. The loyalty discount meets the request in eleven of the twelve attempts, but it respects management's decisions only once. Read the code of one plain-request attempt and say which decisions the agent made: the percentage, the minimum, the cap, the year or the return in progress. *(No GPU.)*

8. Count the false "done"s of each form. The agent's specification and the store's have two each: are they false for the same reason? And why do the plain request and the notes have twice as many? *(No GPU.)*

9. Think of a function in your own work that was asked for as a plain request. Write down its decisions the program does not make: who made them until now, and did the person who programmed it know? *(No GPU.)*

10. **Project.** Choose a real task of your team, talk to the person who does it today and write its notes and its specification, with the decisions the program does not make. Ask a coding agent for it in both ways, as a plain request and with the specification, and compare what it does with each.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). *Continuous Discovery Habits* (Teresa Torres, 2021). The North Cycles store, its program, its tasks and the conversations with its staff are fictional. Original code and experiments.